# Temperature pleasantness

How comfortable a place feels temperature-wise, per 0.5° grid cell. Uses **apparent temperature** (accounts for humidity) from NASA POWER climatology (1991–2020, `T2M` + `RH2M`), scored against a user-tunable comfort band. Ocean cells are masked via `is_land` from `grid.nc`.

The score is deliberately **preference-driven**: some people prefer cool with real seasons, others prefer hot and steady. The pipeline is therefore two-stage:

1. **Stage A (expensive, run once):** fetch monthly T + RH, compute per-month apparent temperature on the atlas grid, cache to `processed/_apparent_temp_monthly.nc`.
2. **Stage B (cheap, rerun any time):** collapse the 12 monthly layers into an annual pleasantness score using two knobs (`ideal_temp`, `tolerance`). This is what `90_scoring.ipynb` consumes.

Changing preferences after the fact does not re-hit the API or re-interpolate — just call `helpers.compute_temperature_pleasantness(ideal_temp, tolerance)`.

In [1]:
import json

import numpy as np
import pandas as pd

from helpers import (
    RAW_DIR,
    PROCESSED_DIR,
    load_grid,
    plot_map,
    save_variable,
    compute_temperature_pleasantness,
)

VARIABLE = 'temperature_pleasantness'
variable_raw = RAW_DIR / VARIABLE
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Fetch raw data

NASA POWER's climatology *regional* endpoint returns monthly + annual means on a 0.5° lat × 0.625° lon grid. The endpoint caps request area at ~10° × 10°, so we tile the globe into 648 tiles and cache each response as JSON. Same pattern as `13_sun_hours.ipynb`, but requesting two parameters (`T2M`, `RH2M`) instead of one.

First run: ~15–25 min (sequential requests). Re-runs: instant (cached).

In [2]:
from common import download_nasa_power_dataset
download_nasa_power_dataset(variable_raw, "T2M")
download_nasa_power_dataset(variable_raw, "RH2M")

POWER tiles: 100%|██████████| 648/648 [11:11<00:00,  1.04s/it]


649 tiles cached in /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/raw/temperature_pleasantness


POWER tiles: 100%|██████████| 648/648 [11:21<00:00,  1.05s/it]

1297 tiles cached in /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/raw/temperature_pleasantness


## 2. Clean & assemble monthly climatology

Each tile JSON is a GeoJSON FeatureCollection. Every feature is a grid point with `geometry.coordinates = [lon, lat]` and `properties.parameter.{T2M,RH2M}.{JAN..DEC, ANN}`. We take all 12 monthly means for both variables. NASA POWER uses `-999` as its fill value — skip those.

In [ ]:
MONTHS = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN',
          'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

records = []
for tile_path in sorted(variable_raw.glob('tile_*_T2M.json')):
    doc = json.loads(tile_path.read_text())
    for feature in doc['features']:
        lon, lat = feature['geometry']['coordinates'][:2]  # ignore elevation
        params = feature['properties']['parameter']
        t = feature['properties']['parameter']['T2M']
        for m_idx, m in enumerate(MONTHS, start=1):
            tv = t.get(m)
            if tv is None or  tv < -900:
                continue
            records.append((lat, lon, m_idx, tv, None))

for tile_path in sorted(variable_raw.glob('tile_*_RH2M.json')):
    doc = json.loads(tile_path.read_text())
    for feature in doc['features']:
        lon, lat = feature['geometry']['coordinates'][:2]  # ignore elevation
        params = feature['properties']['parameter']
        rh = feature['properties']['parameter']['RH2M']
        for m_idx, m in enumerate(MONTHS, start=1):
            rv = rh.get(m)
            if rv is None or rv < -900:
                continue
            records.append((lat, lon, m_idx, None, rv))

# aggregates the dataframe, and groups the two values together
df = (pd.DataFrame(records, columns=['lat', 'lon', 'month', 'T', 'RH'])
    .groupby(['lat', 'lon', 'month'], as_index=False)
    .agg({'T': 'first', 'RH': 'first'}))
df

,lat,lon,month,T,RH
0,0.5,0.625,1,27.04,NaN
1,0.5,0.625,2,27.49,NaN
2,0.5,0.625,3,27.93,NaN
3,0.5,0.625,4,27.87,NaN
4,0.5,0.625,5,27.42,NaN
...,...,...,...,...,...
5519275,-80.0,-170.000,8,NaN,95.24
5519276,-80.0,-170.000,9,NaN,95.51
5519277,-80.0,-170.000,10,NaN,95.87
5519278,-80.0,-170.000,11,NaN,95.45


## 3. Compute apparent temperature and interpolate onto the grid

**Formula (Steadman, simplified, no wind term):**

```
e   = (RH / 100) · 6.105 · exp(17.27 · T / (237.7 + T))   # water vapour pressure, hPa
AT  = T + 0.33 · e − 4.00                                   # apparent temperature, °C
```

NASA POWER also exposes `WS10M`, but the temp/humidity terms dominate for comfort scoring and skipping wind keeps the download small and the formula legible. A future contributor can extend the formula without touching downstream steps.

After computing AT per row we pivot to an xarray `(month, lat, lon)` cube, bilinear-interp onto the 0.5° atlas grid (same method as `13_sun_hours.ipynb`), mask ocean, and cache to `processed/_apparent_temp_monthly.nc`. The underscore prefix signals this file is **not** a scoring layer — it's the frozen climate substrate.

In [ ]:
t = df['T']
e = (df['RH'] / 100.0) * 6.105 * np.exp(17.27 * t / (237.7 + t))
df['AT'] = t + 0.33 * e - 4.00

at_source = (df.set_index(['month', 'lat', 'lon'])['AT']
               .to_xarray()
               .sortby(['month', 'lat', 'lon']))

grid = load_grid()
at_monthly = at_source.interp(lat=grid.lat, lon=grid.lon, method='linear')
at_monthly = at_monthly.where(grid.is_land == 1).astype('float32')
at_monthly.name = 'apparent_temp_monthly'
at_monthly.attrs['units'] = '°C (Steadman apparent temperature, T2M + RH2M, no wind term)'

intermediate = PROCESSED_DIR / '_apparent_temp_monthly.nc'
at_monthly.to_netcdf(intermediate)
print(f'wrote {intermediate}')
at_monthly

## 4. Sanity-check the climate signal

Two quick plots before we score anything:

- **Annual mean apparent temperature.** Should look Köppen-ish — hot near the equator, cold at the poles, deserts warmer than their latitude peers.
- **Annual range** (`max − min` across months). Should peak in continental interiors of the northern hemisphere (Siberia, central Canada) and stay low near coasts and equator.

In [ ]:
annual_mean = at_monthly.mean('month')
annual_mean.name = 'annual mean apparent temperature (°C)'
plot_map(annual_mean, cmap='RdBu_r', robust=True)

In [ ]:
annual_range = at_monthly.max('month') - at_monthly.min('month')
annual_range.name = 'annual apparent-temperature range (°C)'
plot_map(annual_range, cmap='magma', robust=True)

## 5. Preference knobs → pleasantness

Per-month triangular comfort function, then annual mean:

```
comfort(month) = clip(1 − |AT(month) − ideal_temp| / tolerance, 0, 1)
pleasantness   = mean of comfort over 12 months
```

Range 0–1, higher = better. Because pleasantness is applied per month before averaging, a place that sits at 20 °C year-round scores 1.0 while a place that averages 20 °C but swings from −10 to 50 scores near 0 — the whole reason we didn't just use annual mean temperature.

**Tuning notes** (all instant to change, no re-fetch):

- *Prefers cool + real seasons:* drop `IDEAL_TEMP` to ~15 and widen `TOLERANCE` to ~15 so cool summers and mild winters both stay in-band.
- *Prefers tropical, steady heat:* raise `IDEAL_TEMP` to ~25 and tighten `TOLERANCE` to ~5 so seasonal places get penalised.
- *Very picky (Mediterranean only):* `IDEAL_TEMP=20`, `TOLERANCE=5`.

In [ ]:
IDEAL_TEMP = 20.0  # °C — target apparent temperature
TOLERANCE = 10.0   # °C — half-width of the comfort band

pleasantness = compute_temperature_pleasantness(
    ideal_temp=IDEAL_TEMP,
    tolerance=TOLERANCE,
)
pleasantness.name = VARIABLE
pleasantness.attrs['ideal_temp_C'] = IDEAL_TEMP
pleasantness.attrs['tolerance_C'] = TOLERANCE
pleasantness

## 6. Plot pleasantness (and a preference-sensitivity peek)

In [ ]:
plot_map(pleasantness, cmap='RdYlGn', vmin=0, vmax=1,
         title=f'temperature_pleasantness (ideal={IDEAL_TEMP}°C, tolerance={TOLERANCE}°C)')

In [ ]:
# Same intermediate, different preferences — proves the knobs are cheap.
alt = compute_temperature_pleasantness(ideal_temp=25.0, tolerance=5.0)
alt.name = 'temperature_pleasantness (tropical lover: ideal=25°C, tolerance=5°C)'
plot_map(alt, cmap='RdYlGn', vmin=0, vmax=1)

## 7. Save

Writes the default-preference score to `processed/temperature_pleasantness.nc` so `90_scoring.ipynb` picks it up. Re-scoring the atlas for a different preference is a one-liner: `compute_temperature_pleasantness(...).to_netcdf(...)`.

In [ ]:
out = save_variable(pleasantness, VARIABLE)
print(f'wrote {out}')